# Step 1: Sample First 20 Tasks

From `first-20-task-sampling-strategy.md`:
- HotpotQA dev: random 10
- 2WikiMultiHopQA dev: random 10
- No cherry-picking, only filter malformed items
- Goal: test taxonomy stability, not maximize coverage

In [ ]:
!pip install -q datasets

In [ ]:
import random
from datasets import load_dataset

SEED = 42
random.seed(SEED)

## 1. Load HotpotQA dev set

In [ ]:
hotpot = load_dataset("hotpot_qa", "fullwiki", split="validation")
print(f"HotpotQA dev size: {len(hotpot)}")
print(f"Columns: {hotpot.column_names}")
print("---")
print(hotpot[0])

## 2. Load 2WikiMultiHopQA dev set

In [ ]:
wiki2 = load_dataset("scholarly-shadows-syndicate/2WikiMultiHopQA", split="validation")
print(f"2WikiMultiHopQA dev size: {len(wiki2)}")
print(f"Columns: {wiki2.column_names}")
print("---")
print(wiki2[0])

## 3. Random sample 10 + 10

Simple random sampling. Only filter obviously malformed items (empty question or empty answer).

In [ ]:
# --- HotpotQA sampling ---
hp_indices = list(range(len(hotpot)))
random.shuffle(hp_indices)

hp_samples = []
for idx in hp_indices:
    item = hotpot[idx]
    q = item.get("question", "") or ""
    a = item.get("answer", "") or ""
    if len(q.strip()) < 10 or len(a.strip()) == 0:
        continue  # skip malformed
    hp_samples.append({
        "task_id": f"hp_dev_{idx:04d}",
        "dataset": "HotpotQA",
        "original_index": idx,
        "question": q.strip(),
        "answer": a.strip(),
        "type": item.get("type", ""),
        "level": item.get("level", ""),
    })
    if len(hp_samples) == 10:
        break

print(f"HotpotQA sampled: {len(hp_samples)}")

In [ ]:
# --- 2WikiMultiHopQA sampling ---
w2_indices = list(range(len(wiki2)))
random.shuffle(w2_indices)

w2_samples = []
for idx in w2_indices:
    item = wiki2[idx]
    q = item.get("question", "") or ""
    a = item.get("answer", "") or ""
    if len(q.strip()) < 10 or len(a.strip()) == 0:
        continue  # skip malformed
    w2_samples.append({
        "task_id": f"wiki_dev_{idx:04d}",
        "dataset": "2WikiMultiHopQA",
        "original_index": idx,
        "question": q.strip(),
        "answer": a.strip(),
        "type": item.get("type", ""),
    })
    if len(w2_samples) == 10:
        break

print(f"2WikiMultiHopQA sampled: {len(w2_samples)}")

## 4. Review sampled tasks

Look through all 20 tasks. Check:
- Is the question readable?
- Is the answer non-empty?
- Is it actually multi-hop?

Do NOT filter based on difficulty or "typicality".

In [ ]:
all_samples = hp_samples + w2_samples

for i, s in enumerate(all_samples):
    print(f"[{i+1:02d}] {s['task_id']}  ({s['dataset']})")
    print(f"     Q: {s['question']}")
    print(f"     A: {s['answer']}")
    if s.get('type'):
        print(f"     type: {s['type']}")
    print()

## 5. Export to taxonomy.csv

Output matches the schema in `csv-field-examples.md`.

`reasoning_label` and `keep_drop` left blank — you fill these in during the annotation step.

In [ ]:
import csv
import io

output = io.StringIO()
writer = csv.writer(output)
writer.writerow(["task_id", "dataset", "question", "answer", "reasoning_label", "keep_drop", "note"])

for s in all_samples:
    writer.writerow([
        s["task_id"],
        s["dataset"],
        s["question"],
        s["answer"],
        "",  # reasoning_label: fill during annotation
        "",  # keep_drop: fill during annotation
        "",  # note: fill during annotation
    ])

csv_text = output.getvalue()
print(csv_text)

In [ ]:
# Save to file (download from Colab afterwards)
with open("taxonomy_round1_raw.csv", "w", newline="") as f:
    f.write(csv_text)

print("Saved: taxonomy_round1_raw.csv")
print("Download this file, then fill reasoning_label / keep_drop / note columns.")

## 6. Also save the full context (for later source-set construction)

Save supporting paragraphs / evidence so you can inspect reasoning structure during annotation.

In [ ]:
import json

detailed = []
for s in all_samples:
    idx = s["original_index"]
    if s["dataset"] == "HotpotQA":
        item = hotpot[idx]
    else:
        item = wiki2[idx]
    detailed.append({
        "task_id": s["task_id"],
        "dataset": s["dataset"],
        "question": s["question"],
        "answer": s["answer"],
        "raw": {k: v for k, v in item.items()},
    })

with open("sampled_20_full.json", "w") as f:
    json.dump(detailed, f, indent=2, ensure_ascii=False)

print("Saved: sampled_20_full.json")
print("This file contains supporting paragraphs for annotation.")

---

## Next steps

1. Download `taxonomy_round1_raw.csv` and `sampled_20_full.json`
2. Read each question + its supporting paragraphs
3. Fill `reasoning_label` (bridge / comparison / temporal / distractor-heavy) and `keep_drop`
4. Copy the annotated CSV back to `pilot/taxonomy.csv`